<a href="https://colab.research.google.com/github/wjdwogns2873-web/deep-learning-study/blob/main/Kaggle_Study_Practice/12_Cassava_Leaf_Disease_Practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p ~/.kaggle
!cp /content/drive/MyDrive/Kaggle/access_token ~/.kaggle/
!chmod 600 ~/.kaggle/access_token

Mounted at /content/drive


In [2]:
#!/bin/bash
!kaggle datasets download nirmalsankalana/cassava-leaf-disease-classification

Dataset URL: https://www.kaggle.com/datasets/nirmalsankalana/cassava-leaf-disease-classification
License(s): CC0-1.0
100% 2.39G/2.39G [00:24<00:00, 106MB/s]



In [3]:
!unzip -q cassava-leaf-disease-classification.zip -d ./data

In [4]:
!ls -l ./data/data

total 736
drwxr-xr-x 2 root root  36864 Jun 10 14:10 Cassava___bacterial_blight
drwxr-xr-x 2 root root  69632 Jun 10 14:10 Cassava___brown_streak_disease
drwxr-xr-x 2 root root  73728 Jun 10 14:10 Cassava___green_mottle
drwxr-xr-x 2 root root  86016 Jun 10 14:10 Cassava___healthy
drwxr-xr-x 2 root root 487424 Jun 10 14:10 Cassava___mosaic_disease


In [5]:
import os
import torch
from torchvision.datasets import ImageFolder
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, WeightedRandomSampler, random_split
import numpy as np

BATCH_SIZE = 32

data_dir = './data/data'

transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

transform_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

dataset_train = ImageFolder(root=data_dir, transform=transform_train)
dataset_val = ImageFolder(root=data_dir, transform=transform_val)

targets = np.array(dataset_train.targets)
train_size = int(0.8 * len(dataset_train))
val_size = len(dataset_train) - train_size

indices = np.arange(len(dataset_train))
np.random.seed(42)
np.random.shuffle(indices)
train_idx, val_idx = indices[:train_size], indices[train_size:]

train_dataset = torch.utils.data.Subset(dataset_train, train_idx)
val_dataset = torch.utils.data.Subset(dataset_val, val_idx)

train_targets = targets[train_idx]
class_counts = np.bincount(train_targets)
class_weights = 1.0 / class_counts
sample_weights = class_weights[train_targets]
sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

In [6]:
print(len(dataset_train.classes))

5


In [ ]:
import timm
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=len(dataset_train.classes)).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-4)

scheduler = CosineAnnealingLR(optimizer, T_max=3, eta_min=1e-6)

In [8]:
import torch.nn as nn
import torch.nn.functional as F

class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)

        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean()

criterion = FocalLoss(alpha=1, gamma=2).to(device)

In [10]:
from sklearn.metrics import f1_score

epochs = 3

for epoch in range(epochs):
    model.train()
    train_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)

    model.eval()
    val_loss = 0.0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)

            preds = outputs.argmax(1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    epoch_train_loss = train_loss / len(train_dataset)
    epoch_val_loss = val_loss / len(val_dataset)
    epoch_macro_f1 = f1_score(all_labels, all_preds, average='macro')

    current_lr = scheduler.get_last_lr()[0]

    scheduler.step()

    print(f"Epoch [{epoch+1}/{epochs}] | LR: {current_lr:.6f} | "\
          f"Train Loss: {epoch_train_loss:4f} | Val Loss: {epoch_val_loss:.4f} | Val Macro F1: {epoch_macro_f1:.4f}")

Epoch [1/3] | LR: 0.000100 | Train Loss: 0.581658 | Val Loss: 0.6150 | Val Macro F1: 0.5605
Epoch [2/3] | LR: 0.000075 | Train Loss: 0.380399 | Val Loss: 0.4988 | Val Macro F1: 0.6010
Epoch [3/3] | LR: 0.000026 | Train Loss: 0.312114 | Val Loss: 0.4308 | Val Macro F1: 0.6134
